# Block 4 transfer

> Transfer the pretrained Kex model from block 4 instead of from the whole network: reset block 5, cut after block 4 with a pooled or a flattened head, and start the flattened head from a ridge regression

In [ ]:
#| default_exp variants

In [ ]:
#| export
import copy
from soilspecdl.all import SpectralCNN

/Users/franckalbinet/pro/dev/soilspec/ossl-fuku-transfer/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from fastcore.test import *
from fastai.torch_core import set_seed
from soilspecdl.all import SpectralCNN, CutCNN, ridge_head, predict
from soilspecdata.all import load_ossl
from ossl_fuku_transfer.data import *
from ossl_fuku_transfer.transfer import *
from ossl_fuku_transfer.inside import layer_features, probe

`06_dynamics` and `07_inside_transfer` found that block 5 and the global pool are the part of the pretrained model most specific to OSSL, and that block 4 holds the most Kex information on Fukushima. The variants here test whether transferring from block 4 does better than fine-tuning the whole network.

The module part uses small random arrays, so it runs without the data. The sections from *Data* onwards read OSSL and the Fukushima data, and are marked `#| fukudata`.

## Variants

The variants use soilspecdl's `CutCNN`, which keeps the first blocks of a `SpectralCNN` and adds a new head. Its head reads either the average of the last block's output over positions (`flat=False`), as `SpectralCNN` does, or all channels at all positions (`flat=True`), which keeps band position. soilspecdl's `ridge_head` fits a ridge regression from a model's frozen features to the targets, and copies it into the head. Fine-tuning then starts from a head that already reads the frozen features well, instead of from a random one.


`reset_top` keeps all blocks but gives block 5 and the head new random weights:

In [ ]:
#| export
def reset_top(
    model, # `SpectralCNN`, such as one returned by `pretrain`
):
    "Copy of `model` whose last block and head have the random weights of a new `SpectralCNN`"
    m, fresh = copy.deepcopy(model), SpectralCNN()
    m.backbone.layers[-1].load_state_dict(fresh.backbone.layers[-1].state_dict())
    m.head.load_state_dict(fresh.head.state_dict())
    return m

In [ ]:
m0 = SpectralCNN()
m1 = reset_top(m0)
test_eq(torch.equal(m1.backbone.layers[3].conv[0].weight, m0.backbone.layers[3].conv[0].weight), True)
test_eq(torch.equal(m1.backbone.layers[4].conv[0].weight, m0.backbone.layers[4].conv[0].weight), False)

## Data

The data and preprocessing are those of `03_kex`, with the saved pretrained Kex model:

In [ ]:
#| fukudata
om = load_ossl().mir(wmin=shared_wns[0], wmax=shared_wns[-1])
Xo, yo, _ = om.xy('k.ext_usda.a725_cmolc.kg')
yo = np.log1p(yo.ravel())
fuku = load_fuku()
fuku.source = fuku.source.assign(target=fuku.source['soil_ex_K2O'] / mg_k2o_per_cmolc)
Xf, yf, idf = fuku.xy('target')
yf = np.log1p(yf.ravel())
is_test = np.isin(idf, cs137_split()[1])
yt = yf[is_test]
pipe = preprocessings()['snv_d1']
Xo_d, Xf_d = pipe.fit_transform(Xo), pipe.transform(Xf)
pretrained = load_pretrained('ossl_kex_snv_d1')
int((~is_test).sum()), int(is_test.sum())

(1047, 341)

## Comparison

Five starting points, each fine-tuned end to end with a one-cycle schedule of 20 epochs:

- `full`: the whole pretrained model, as `fully fine-tuned` in `03_kex`;
- `block 5 reset`: the pretrained model with new random weights in block 5 and the head;
- `cut, pooled`: blocks 1 to 4 with a new head on their average over positions;
- `cut, flat`: blocks 1 to 4 with a new head on all channels and positions;
- `cut, flat, ridge head`: the same, with the head set by `ridge_head` on the drawn training samples.

`ridge, block 4` is the frozen reference of `08_learning_curve`: a ridge regression on block 4's flattened output, without fine-tuning. Each starting point trains at two maximum learning rates, 3·10⁻⁴ and 3·10⁻³, on 400 and on all 1,047 training samples, with three seeds. `subset_preds` draws the samples, so all starting points of a seed and size use the same ones. The results are saved in `model_dir()` and loaded on later runs:

In [ ]:
#| fukudata
starts = {'full': pretrained,
          'block 5 reset': lambda sub: reset_top(pretrained),
          'cut, pooled': lambda sub: CutCNN(pretrained, len(shared_wns)),
          'cut, flat': lambda sub: CutCNN(pretrained, len(shared_wns), flat=True),
          'cut, flat, ridge head': lambda sub: ridge_head(CutCNN(pretrained, len(shared_wns), flat=True), Xf_d[sub], yf[sub], shared_wns)}
F4 = layer_features(pretrained, Xf_d)['block 4']
sizes, seeds, lrs = [400, 1047], range(3), [3e-4, 3e-3]
fn = model_dir()/'variants_kex.csv'
if fn.exists(): var = pd.read_csv(fn)
else:
    rows = []
    for n in sizes:
        for s in seeds:
            sub = np.random.RandomState(s).choice(np.flatnonzero(~is_test), n, replace=False)
            res = probe(F4[sub], yf[sub], F4[is_test], yt)
            rows.append({'variant': 'ridge, block 4', 'lr': 0, 'n': n, 'seed': s, **{k: v for k, v in res.items() if k != 'alpha'}})
            for v, st in starts.items():
                for lr in lrs:
                    rows.append({'variant': v, 'lr': lr, 'n': n, 'seed': s, **scores(yt, subset_preds(st, Xf_d, yf, is_test, n, s, 20, lr))})
    var = pd.DataFrame(rows)
    var.to_csv(fn, index=False)
summ = var.groupby(['n', 'variant', 'lr'], sort=False)[['R²', 'r']].agg(['mean', 'std'])
summ.round(3)

R²             r       
                                    mean    std   mean    std
n    variant               lr                                
400  ridge, block 4        0.0000  0.345  0.012  0.589  0.009
     full                  0.0003  0.195  0.026  0.464  0.021
                           0.0030  0.254  0.033  0.529  0.027
     block 5 reset         0.0003  0.201  0.031  0.456  0.028
                           0.0030  0.250  0.030  0.513  0.035
     cut, pooled           0.0003  0.034  0.024  0.200  0.071
                           0.0030  0.142  0.034  0.391  0.036
     cut, flat             0.0003  0.203  0.010  0.453  0.013
                           0.0030  0.290  0.017  0.548  0.013
     cut, flat, ridge head 0.0003  0.315  0.008  0.565  0.004
                           0.0030  0.308  0.020  0.560  0.016
1047 ridge, block 4        0.0000  0.429  0.000  0.657  0.000
     full                  0.0003  0.329  0.016  0.574  0.014
                           0.0030  0.410  0.021  0.645  0.013
     block 5 reset         0.0003  0.264  0.015  0.524  0.012
                           0.0030  0.383  0.020  0.619  0.016
     cut, pooled           0.0003  0.092  0.012  0.353  0.023
                           0.0030  0.230  0.007  0.486  0.007
     cut, flat             0.0003  0.295  0.006  0.552  0.008
                           0.0030  0.367  0.011  0.608  0.010
     cut, flat, ridge head 0.0003  0.380  0.007  0.622  0.007
                           0.0030  0.384  0.013  0.621  0.011

Mean test R² of each starting point at its better learning rate, with the standard deviation over the seeds:

In [ ]:
#| fukudata
best = summ.loc[summ[('R²', 'mean')].groupby(level=[0, 1], sort=False).idxmax()]
best.round(3)

R²             r       
                                    mean    std   mean    std
n    variant               lr                                
400  ridge, block 4        0.0000  0.345  0.012  0.589  0.009
     full                  0.0030  0.254  0.033  0.529  0.027
     block 5 reset         0.0030  0.250  0.030  0.513  0.035
     cut, pooled           0.0030  0.142  0.034  0.391  0.036
     cut, flat             0.0030  0.290  0.017  0.548  0.013
     cut, flat, ridge head 0.0003  0.315  0.008  0.565  0.004
1047 ridge, block 4        0.0000  0.429  0.000  0.657  0.000
     full                  0.0030  0.410  0.021  0.645  0.013
     block 5 reset         0.0030  0.383  0.020  0.619  0.016
     cut, pooled           0.0030  0.230  0.007  0.486  0.007
     cut, flat             0.0030  0.367  0.011  0.608  0.010
     cut, flat, ridge head 0.0030  0.384  0.013  0.621  0.011

## Findings

No fine-tuned starting point beats the frozen reference. With 1,047 training samples, the ridge regression on block 4 gives a test R² of 0.43 (r 0.66). The best fine-tuned model is the whole pretrained model, at 0.41 (r 0.65). With 400 samples, the ridge regression gives 0.35, and the best fine-tuned model is the flattened cut with a ridge head, at 0.32.

Fine-tuning the flattened cut with a ridge head lowers its score. Before fine-tuning, it predicts exactly what the reference predicts: its head is the same ridge regression on the same features and samples. After fine-tuning, it scores 0.38 with 1,047 samples and 0.31 to 0.32 with 400, at either learning rate. With 1,047 samples, it ends level with resetting block 5 (0.38). The two differ from the reference in the samples they see: the ridge head is fitted on all drawn samples, and fine-tuning trains on 90% of them. That alone is unlikely to explain a drop of 0.05.

On block 4, keeping band position matters. A new head on all channels and positions of block 4 gives 0.37 with 1,047 samples. A new head on their average over positions gives 0.23. With 400 samples, the two give 0.29 and 0.14. Averaging block 4's output over positions loses much of its Kex information.

Block 5's pretrained weights help fine-tuning. Resetting block 5 and the head gives 0.38 with 1,047 samples, against 0.41 for the whole pretrained model. So block 5 is specific to OSSL as a frozen layer, as `07_inside_transfer` found, but it is still a better starting point than random weights.

The higher learning rate, 3·10⁻³, is better for every starting point except the ridge head, where the two rates give the same result within the spread over seeds.

With these schedules, the best use of the pretrained model on Fukushima Kex is as a frozen feature extractor read by a ridge regression on block 4, not as a starting point for fine-tuning. Fine-tuning moves the model away from a readout that is already better than what it learns.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()